In [0]:
from pyspark.sql import functions as F

silver_df = spark.table("fraud_detection.silver.transactions_enriched")

model_df = silver_df.select(
    "transaction_id",
    "is_fraudulent",
    "price_out_of_range",
    "is_blocklisted",
    "blacklist_count",
    "currency_is_defaulted",
    "fx_rate_is_defaulted",
    "transaction_amount_usd",
    "customer_age",
    "account_age_days",
    "transaction_hour"
).na.fill(0)

display(model_df.limit(10))
print(f"Row count: {model_df.count()}")

In [0]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

pdf = model_df.toPandas()

feature_cols = [
    "price_out_of_range", "is_blocklisted", "blacklist_count",
    "currency_is_defaulted", "fx_rate_is_defaulted",
    "transaction_amount_usd", "customer_age", "account_age_days", "transaction_hour"
]

# convert booleans to 0/1
for col in ["price_out_of_range", "is_blocklisted", "currency_is_defaulted", "fx_rate_is_defaulted"]:
    pdf[col] = pdf[col].astype(int)

X = pdf[feature_cols]
y = pdf["is_fraudulent"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")
print(f"Fraud rate in train: {y_train.mean():.4f}, in test: {y_test.mean():.4f}")

In [0]:
model = LogisticRegression(max_iter=1000, class_weight="balanced")
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_proba = model.predict_proba(X_test)[:, 1]

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1: {f1:.4f}")
print(f"ROC-AUC: {auc:.4f}")
print(f"\nConfusion Matrix:\n{confusion_matrix(y_test, y_pred)}")

In [0]:
import pandas as pd

coef_df = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": model.coef_[0]
}).sort_values("coefficient", ascending=False)

print(coef_df)

In [0]:
test_ids = pdf.loc[X_test.index, "transaction_id"]

heuristic_test_df = spark.sql(f"""
    SELECT transaction_id, risk_tier, is_fraudulent
    FROM fraud_detection.gold.transactions_risk_scored
    WHERE transaction_id IN ({','.join([f"'{i}'" for i in test_ids])})
""").toPandas()

heuristic_pred = (heuristic_test_df["risk_tier"] == "high").astype(int)
heuristic_actual = heuristic_test_df["is_fraudulent"]

from sklearn.metrics import precision_score, recall_score, f1_score

print(f"Heuristic Precision: {precision_score(heuristic_actual, heuristic_pred):.4f}")
print(f"Heuristic Recall: {recall_score(heuristic_actual, heuristic_pred):.4f}")
print(f"Heuristic F1: {f1_score(heuristic_actual, heuristic_pred):.4f}")

In [0]:
import joblib

# Save predictions alongside actuals for the full test set
results_df = X_test.copy()
results_df["transaction_id"] = pdf.loc[X_test.index, "transaction_id"]
results_df["actual_fraud"] = y_test
results_df["predicted_fraud"] = y_pred
results_df["predicted_probability"] = y_pred_proba

results_spark_df = spark.createDataFrame(results_df)
results_spark_df.write.mode("overwrite").saveAsTable("fraud_detection.gold.model_predictions")

display(spark.sql("SELECT COUNT(*) FROM fraud_detection.gold.model_predictions"))